# Flight Delay Analysis
Portfolio copy of my project notebook. Outputs are removed for a cleaner GitHub view; the code below comes from the original analysis.

In [ ]:
# Creating a controllable delay binary variable
df['controllable_delay_binary'] = (
    (df['carrier_delay'] > 0) |
    (df['nas_delay'] > 0) |
    (df['security_delay'] > 0)
).astype(int)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import shap


In [ ]:
xgb_scale = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=(train_y == 0).sum() / (train_y == 1).sum(),
    random_state=1
)
xgb_scale.fit(train_x, train_y)

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, precision_score, f1_score
import pandas as pd
model_list = [
    ('Decision Tree Balanced', df_tree_balanced),
    ('Decision Tree Unbalanced', df_tree_unbalanced),
    ('Random Forest Balanced', rf_balanced),
    ('Random Forest Unbalanced', rf_unbalanced),
    ('Logistic Regression Balanced', log_balanced),
    ('Logistic Regression Unbalanced', log_unbalanced),
    ('XGBoost Scaled', xgb_scale),
    ('XGBoost Unscaled', xgb_unscale)
]
results = []
for name, model in model_list:
    preds = model.predict(valid_x)
    results.append({'Model': name, 'Accuracy': accuracy_score(valid_y, preds), 'Balanced_Accuracy': balanced_accuracy_score(valid_y, preds), 'Recall': recall_score(valid_y, preds), 'Precision': precision_score(valid_y, preds, zero_division=0), 'F1_Score': f1_score(valid_y, preds, zero_division=0)})
summary_df = pd.DataFrame(results).round(3).sort_values(by='F1_Score', ascending=False).reset_index(drop=True)
summary_df

In [ ]:
explainer = shap.Explainer(xgb_scale, train_x.astype(float))
shap_values = explainer(valid_x.sample(100, random_state=1).astype(float))
shap.plots.beeswarm(shap_values, max_display=10)